In [34]:
import wandb
import pandas as pd

# Initialize W&B API
api = wandb.Api()

# Set your entity and project name
ENTITY = "ahmed-attia-mbzuai"   # W&B username or team name
PROJECT = "sampling-strategies" # W&B project name

# Get all runs
runs = api.runs(f"{ENTITY}/{PROJECT}")

# Extract relevant data
runs_data = []
for run in runs:
    runs_data.append({
        "id": run.id,
        "config": run.config,  # Run configurations (parameters)
        "summary": run.summary._json_dict,  # Final metrics
    })

In [35]:
# Convert to DataFrame
df = pd.DataFrame(runs_data)

In [36]:
# df normalize config and summart columns
df = pd.concat([df.drop(['config'], axis=1), df['config'].apply(pd.Series)], axis=1)
df = pd.concat([df.drop(['summary'], axis=1), df['summary'].apply(pd.Series)], axis=1)

In [37]:
df

,id,R0,R1,R2,lr,loss,lr_Z,ndim,plot,seed,...,validation_interval,replay_buffer_prioritized,_runtime,_step,_timestamp,_wandb,l1_dist,logZ_diff,loss,states_visited
0,55m4kxw8,0.00010,1,3,0.001,TB,0.1,2,False,0,...,100,False,131.201540,12499,1.743160e+09,{'runtime': 131},0.015278,0.002941,4.859976e-03,200000
1,dn48yqkf,0.01000,1,3,0.001,TB,0.1,2,False,0,...,100,False,231.961165,12499,1.743160e+09,{'runtime': 231},0.000473,0.000144,4.238900e-03,200000
2,dy73eo9w,0.00001,1,3,0.001,TB,0.1,2,False,0,...,100,False,93.292623,12499,1.743160e+09,{'runtime': 93},0.023437,1.386417,1.932044e-09,200000
3,i7awgb1w,0.00100,1,3,0.001,TB,0.1,2,False,0,...,100,False,221.922686,12499,1.743160e+09,{'runtime': 221},0.002436,0.009284,6.617450e-05,200000
4,ipumy5tq,0.00050,1,3,0.001,TB,0.1,2,False,0,...,100,False,207.326961,12499,1.743160e+09,{'runtime': 207},0.004111,0.000648,1.164991e-03,200000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211,hrhzxtw5,0.05000,1,3,0.001,TB,0.1,2,False,0,...,100,False,4378.498712,12499,1.743231e+09,{'runtime': 4378},0.000022,0.185877,1.210396e-01,200000
212,l945g9qg,0.10000,1,3,0.001,TB,0.1,2,False,0,...,100,False,3863.356650,12499,1.743231e+09,{'runtime': 3863},0.000022,0.107582,5.679680e-02,200000
213,b3bk6g5u,0.10000,1,3,0.001,TB,0.1,2,False,0,...,100,False,1735.818719,12499,1.743233e+09,{'runtime': 1735},0.000016,0.022255,3.462856e-02,200000
214,czk5m07v,0.05000,1,3,0.001,TB,0.1,2,False,0,...,100,False,1718.836845,12499,1.743233e+09,{'runtime': 1718},0.000016,0.034320,2.596633e-02,200000


In [38]:
# pick only R0, R1, R2, l1_dist, sampler columns 

df = df[['R0', 'R1', 'R2', 'l1_dist', 'sampler', "height", "ndim", "lr", "lr_Z", "id", "n_trajectories"]]

In [39]:
import seaborn as sns
import os
import matplotlib.pyplot as plt
# Create folder if it doesn't exist
dir = "rewards_vs_l1dist_different"
os.makedirs(dir, exist_ok=True)
unique_heights = df['height'].unique()
unique_ndims = df['ndim'].unique()
for height in unique_heights:
    for ndim in unique_ndims:
        subset = df[(df['height'] == height) & (df['ndim'] == ndim)]
        if subset.empty:
            continue
        plt.figure(figsize=(8, 6))
        ax = sns.lineplot(x='R0', y='l1_dist', hue='sampler', data=subset)
        ax.set_title(f'Height {height} | ndim {ndim}')
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_xlabel('R0')
        ax.set_ylabel('L1 Distance')
        plt.tight_layout()
        plt.savefig(f"{dir}/plot_height_{height}_ndim_{ndim}.png")
        plt.close()
